# USD/CHF (DEXSZUS): GARCH and Markov-switching volatility

Copy of the USD/JPY notebook with every model cell pointed at the FRED series `DEXSZUS`. Cells before the expanding-window GARCH cell (data loading and the cross-pair overview) are unchanged. Variable names keep their `jpy_` prefixes so the cells still link up; they refer to USD/CHF here.

In [ ]:
'''import pandas as pd

# Load the datasets
vix_df = pd.read_csv('C:/Users/guosh/OneDrive/Desktop/mfams/VIXCLS_volatility.csv')
# fx_df = pd.read_csv('C:/Users/guosh/OneDrive/Desktop/mfams/FXdataDaily_2005_to_2026_Dev_and_underdeve/daily.csv')

# Display the first few rows of each
print("VIX Volatility Data (first 5 rows):")
print(vix_df.head())

#print("\nFX Daily Data (first 5 rows):")
#print(fx_df.head())'''

In [ ]:
import pandas as pd
fx_series = [
    'DEXBZUS', 'DEXCAUS', 'DEXCHUS', 'DEXDNUS', 'DEXHKUS', 'DEXINUS',
    'DEXJPUS', 'DEXKOUS', 'DEXMAUS', 'DEXMXUS', 'DEXNOUS', 'DEXSDUS',
    'DEXSFUS', 'DEXSIUS', 'DEXSLUS', 'DEXSZUS', 'DEXTAUS', 'DEXTHUS',
    'DEXUSAL', 'DEXUSEU', 'DEXUSNZ', 'DEXUSUK', 'DEXVZUS'
]

start_date = '2005-01-01'
end_date = pd.Timestamp.today().strftime('%Y-%m-%d')
fx_df = None

for series in fx_series:
    url = (
        f'https://fred.stlouisfed.org/graph/fredgraph.csv'
        f'?id={series}&cosd={start_date}&coed={end_date}'
    )
    series_df = pd.read_csv(url, na_values='.')
    series_df.columns = ['observation_date', series]
    series_df[series] = pd.to_numeric(series_df[series], errors='coerce')

    fx_df = series_df if fx_df is None else fx_df.merge(
        series_df, on='observation_date', how='outer'
    )

fx_df['observation_date'] = pd.to_datetime(fx_df['observation_date'])
fx_df = fx_df.sort_values('observation_date').reset_index(drop=True)

display(fx_df.head())
print(f'Date range: {fx_df["observation_date"].min().date()} to '
      f'{fx_df["observation_date"].max().date()}')

In [ ]:
start_date = '2005-01-01'
end_date = pd.Timestamp.today().strftime('%Y-%m-%d')

vix_url = (
    'https://fred.stlouisfed.org/graph/fredgraph.csv'
    f'?id=VIXCLS&cosd={start_date}&coed={end_date}'
)

vix_df = pd.read_csv(vix_url, na_values='.')
vix_df['observation_date'] = pd.to_datetime(vix_df['observation_date'])
vix_df['VIXCLS'] = pd.to_numeric(vix_df['VIXCLS'], errors='coerce')

display(vix_df.head())
print(f'Date range: {vix_df["observation_date"].min().date()} to '
      f'{vix_df["observation_date"].max().date()}')

In [ ]:
# Convert VIXCLS column to numeric, handling missing or non-numeric values (coercing them to NaN)
vix_df['VIXCLS'] = pd.to_numeric(vix_df['VIXCLS'], errors='coerce')

# Filter the rows based on the threshold of 30
High_volatility = vix_df[vix_df['VIXCLS'] > 30].copy()
Medium_volatility = vix_df[(vix_df['VIXCLS'] <= 30) & (vix_df['VIXCLS'] > 15)].copy()
Low_volatility = vix_df[vix_df['VIXCLS'] <= 15].copy()

# Display the sizes and preview of each dataset
print(f"High Volatility shape: {High_volatility.shape}")
print(f"Medium Volatility shape: {Medium_volatility.shape}")
print(f"Low Volatility shape: {Low_volatility.shape}")

print("\nHigh Volatility Preview:")
display(High_volatility.head())

print("\nMedium Volatility Preview:")
display(Medium_volatility.head())

print("\nLow Volatility Preview:")
display(Low_volatility.head())

direction: interpret the garch and the markov models + find ways to optimize them? see which is better in jpy/usd and in another type of currency perhaps 

Other safe havens

USD/CHF: the classic second safe haven. Swiss franc strength tends to track JPY in risk-off episodes, so it’s a useful check on whether JPY’s behavior is special or part of a broader safe-haven pattern.
EUR/USD: the most liquid pair, and a useful “neutral major” baseline. It has some safe-haven flavor, but it usually trades as a mix of risk and rate-differential effects.

Commodity / risk-on currencies

AUD/USD: the textbook risk-on currency, sensitive to global growth and China. It’s usually the sharpest contrast to JPY in turbulent regimes.
NZD/USD: similar to AUD, with higher idiosyncratic noise.
USD/CAD: tied to oil. It helps separate commodity-driven moves from pure risk sentiment.

Other majors

GBP/USD: a large, liquid pair with its own event-driven spikes (Brexit in 2016 is a natural turbulence episode to examine).

Emerging-market / high-beta currencies

USD/MXN: the most liquid EM pair and a common risk proxy. It tends to show large volatility jumps and wide spreads in stress.
USD/ZAR: very high beta to risk sentiment and commodity prices.
USD/TRY: structurally volatile with its own crisis regime (2018, 2021 onward), so it will probably dominate any cross-sectional comparison. It’s worth including, but consider analyzing it separately.
USD/BRL, USD/INR or USD/KRW: optional additional EM exposure. Check whether forexsb covers these, since its EM coverage is thinner.

Suggested minimal set: USD/JPY, USD/CHF, EUR/USD, AUD/USD, USD/CAD, USD/MXN, USD/ZAR. That gives you two safe havens, a neutral major, two commodity currencies, and two EM currencies, which is enough to see clean differences across categories without drowning in pairs.

In [ ]:
from arch import arch_model
import matplotlib.pyplot as plt
import numpy as np

# Create a copy of the FX dataframe
fx_calc = fx_df.copy()

# Set observation_date as the index
if 'observation_date' in fx_calc.columns:
    fx_calc.set_index('observation_date', inplace=True)

# Ensure all currency exchange rate columns are numeric
for col in fx_calc.columns:
    fx_calc[col] = pd.to_numeric(fx_calc[col], errors='coerce')

# Calculate daily percentage returns
fx_returns = fx_calc.pct_change()
# Define the representative currency pairs we want to analyze
selected_pairs = ['DEXJPUS', 'DEXMXUS', 'DEXUSEU']

# We will store forecast errors for each pair under different VIX regimes
comparison_results = {}

# Identify regime dates based on our VIX-based dataframes
high_vol_dates = set(High_volatility['observation_date'].values)
med_vol_dates = set(Medium_volatility['observation_date'].values)
low_vol_dates = set(Low_volatility['observation_date'].values)

for pair in selected_pairs:
    # Clean and scale returns to percentage
    returns_series = fx_returns[pair].dropna() * 100

    # Fit GARCH(1,1) model
    model = arch_model(returns_series, vol='Garch', p=1, q=1, dist='normal')
    fit_res = model.fit(disp='off')

    # Volatility forecast error:
    # Realized Volatility Proxy (Squared returns) minus GARCH predicted Conditional Variance
    realized_var = returns_series ** 2
    garch_var = fit_res.conditional_volatility ** 2
    forecast_error = (realized_var - garch_var)

    # Align forecast errors back to their original observation dates
    fe_df = pd.DataFrame({'Forecast_Error': forecast_error}, index=returns_series.index)

    # Segment the forecast errors into the VIX-based regimes
    high_errors = fe_df.loc[fe_df.index.isin(high_vol_dates), 'Forecast_Error'].dropna()
    med_errors = fe_df.loc[fe_df.index.isin(med_vol_dates), 'Forecast_Error'].dropna()
    low_errors = fe_df.loc[fe_df.index.isin(low_vol_dates), 'Forecast_Error'].dropna()

    comparison_results[pair] = {
        'High Vol Regime Mean Absolute Error': np.abs(high_errors).mean(),
        'Med Vol Regime Mean Absolute Error': np.abs(med_errors).mean(),
        'Low Vol Regime Mean Absolute Error': np.abs(low_errors).mean(),
        'High Vol Regime RMSE': np.sqrt((high_errors ** 2).mean()),
        'Low Vol Regime RMSE': np.sqrt((low_errors ** 2).mean())
    }

    # Convert results to DataFrame for display
comparison_errors_df = pd.DataFrame(comparison_results).T
display(comparison_errors_df)

# Plot the comparison of Root Mean Squared Volatility Forecast Errors
comparison_errors_df[['High Vol Regime RMSE', 'Low Vol Regime RMSE']].plot(kind='bar', figsize=(10, 6))
plt.title('GARCH(1,1) Volatility Forecast Error (RMSE) across Volatility Regimes')
plt.ylabel('Root Mean Squared Error (RMSE)')
plt.xlabel('FX Pair')
plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Expanding-window real-time GARCH(1,1) forecasts for DEXSZUS.
# Estimate on 2005-2006, then refit each day using only returns available
# before that day's forecast; add the observed return after forecasting.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from arch import arch_model

expanding_fx = fx_df[['observation_date', 'DEXSZUS']].copy()
expanding_fx['observation_date'] = pd.to_datetime(expanding_fx['observation_date'], errors='coerce')
expanding_fx['DEXSZUS'] = pd.to_numeric(expanding_fx['DEXSZUS'], errors='coerce')
expanding_fx = expanding_fx.dropna().set_index('observation_date').sort_index()
expanding_returns = expanding_fx['DEXSZUS'].pct_change(fill_method=None).dropna() * 100

initial_returns = expanding_returns.loc['2005-01-01':'2006-12-31']
forecast_returns = expanding_returns.loc['2007-01-01':'2026-12-31']
if initial_returns.empty or forecast_returns.empty:
    raise ValueError('DEXSZUS data must include returns in both 2005-2006 and 2007-2026.')

expanding_history = initial_returns.copy()
expanding_forecast_variances = pd.Series(
    index=forecast_returns.index,
    dtype=float,
    name='One-day-ahead GARCH forecast variance'
)
expanding_garch_rows = []

for step, (forecast_date, observed_return) in enumerate(forecast_returns.items(), start=1):
    expanding_model = arch_model(
        expanding_history,
        mean='Constant',
        vol='Garch',
        p=1,
        q=1,
        dist='normal'
    )
    expanding_fit = expanding_model.fit(disp='off')
    forecast_variance = expanding_fit.forecast(horizon=1, reindex=False).variance.iloc[-1, 0]
    expanding_forecast_variances.loc[forecast_date] = forecast_variance

    # Log this refit's parameters
    pr = expanding_fit.params
    om, al, be = pr['omega'], pr['alpha[1]'], pr['beta[1]']
    pers = al + be
    expanding_garch_rows.append({
        'date': forecast_date, 'omega': om, 'alpha': al, 'beta': be, 'persistence': pers,
        'halflife_days': np.log(0.5) / np.log(pers) if 0 < pers < 1 else np.nan,
        'longrun_vol_ann': np.sqrt(om / (1 - pers) * 252) if pers < 1 else np.nan,
    })

    # Include today's observed return only after its forecast has been recorded.
    expanding_history.loc[forecast_date] = observed_return

    if step % 500 == 0:
        print(f'Refitted through forecast {step:,} of {len(forecast_returns):,} ({forecast_date.date()}).')

expanding_garch_log = pd.DataFrame(expanding_garch_rows).set_index('date')
expanding_forecast_volatility = np.sqrt(expanding_forecast_variances) * np.sqrt(252)
expanding_realized_volatility = (
    expanding_returns.rolling(window=30, min_periods=30).std() * np.sqrt(252)
).reindex(forecast_returns.index)

expanding_vix = vix_df[['observation_date', 'VIXCLS']].copy()
expanding_vix['observation_date'] = pd.to_datetime(expanding_vix['observation_date'], errors='coerce')
expanding_vix['VIXCLS'] = pd.to_numeric(expanding_vix['VIXCLS'], errors='coerce')
expanding_vix = expanding_vix.dropna().set_index('observation_date')['VIXCLS'].sort_index()
expanding_vix = expanding_vix.reindex(forecast_returns.index, method='ffill')
if expanding_vix.notna().sum() == 0:
    raise ValueError('No VIX observations overlap the GARCH forecast period.')

expanding_vix_regime = pd.Series(
    np.select([expanding_vix > 30, expanding_vix > 15], ['High', 'Medium'], default='Low'),
    index=expanding_vix.index,
).where(expanding_vix.notna())
expanding_regime_colors = {'Low': 'tab:green', 'Medium': 'gold', 'High': 'tab:red'}
expanding_regime_groups = expanding_vix_regime.ne(expanding_vix_regime.shift()).cumsum()


fig, ax = plt.subplots(figsize=(15, 6))
for _, regime_dates in expanding_vix_regime.groupby(expanding_regime_groups):
    regime = regime_dates.iloc[0]
    if pd.notna(regime):
        ax.axvspan(
            regime_dates.index[0],
            regime_dates.index[-1] + pd.Timedelta(days=1),
            color=expanding_regime_colors[regime],
            alpha=0.12,
            linewidth=0,
            zorder=0,
        )
ax.plot(
    expanding_forecast_volatility.index,
    expanding_forecast_volatility,
    color='tab:blue',
    linewidth=1,
    label='Daily-refit expanding-window GARCH forecast (annualized)'
)
ax.plot(
    expanding_realized_volatility.index,
    expanding_realized_volatility,
    color='tab:orange',
    linewidth=1,
    alpha=0.8,
    label='Trailing 30-day realized volatility (annualized)'
)
ax.set_title('DEXSZUS: Expanding-Window Real-Time GARCH Forecasts (2007-2026)')
ax.set_xlabel('Forecast date')
ax.set_ylabel('Annualized volatility (%)')
ax.grid(True, linestyle=':', alpha=0.5)
line_handles, line_labels = ax.get_legend_handles_labels()
regime_handles = [
    Patch(facecolor=expanding_regime_colors[name], alpha=0.2, label=f'{name} VIX regime')
    for name in ['Low', 'Medium', 'High']
]
ax.legend(handles=line_handles + regime_handles,
          labels=line_labels + [handle.get_label() for handle in regime_handles])
fig.tight_layout()
plt.show()

print(f'Initial fit period: {initial_returns.index.min().date()} to {initial_returns.index.max().date()}')
print(f'Forecast period: {forecast_returns.index.min().date()} to {forecast_returns.index.max().date()}')
print(f'Daily refits/forecasts: {expanding_forecast_volatility.notna().sum():,}')

In [ ]:
# ===== GARCH(1,1) parameters: expanding window =====
def show_garch_params(log, label):
    cols = ['omega', 'alpha', 'beta', 'persistence', 'halflife_days']
    last = log.iloc[-1]
    print(f'{label} - latest refit ({log.index[-1].date()}):')
    print(f'  omega = {last["omega"]:.5f} | alpha = {last["alpha"]:.4f} | beta = {last["beta"]:.4f} | '
          f'alpha+beta = {last["persistence"]:.4f} | half-life = {last["halflife_days"]:.1f} days')
    print(f'\nSummary over all {len(log):,} refits:')
    display(log[cols].describe().T[['mean', 'std', 'min', '50%', 'max']].round(4))
    print('Parameters at the last refit of each year:')
    yearly = log[cols].groupby(log.index.year).last()
    yearly.index.name = 'year'
    display(yearly.round(4))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
    ax1.plot(log.index, log['alpha'], color='tab:blue', linewidth=1)
    ax1.set_ylabel('alpha')
    ax1.set_title(f'{label}: GARCH(1,1) parameters at each daily refit')
    ax1.grid(True, linestyle=':', alpha=0.5)
    ax2.plot(log.index, log['beta'], color='tab:orange', linewidth=1, label='beta')
    ax2.plot(log.index, log['persistence'], color='tab:red', linewidth=1, label='alpha + beta')
    ax2.set_ylabel('beta, alpha + beta')
    ax2.set_xlabel('Forecast date')
    ax2.grid(True, linestyle=':', alpha=0.5)
    ax2.legend()
    fig.tight_layout()
    plt.show()

show_garch_params(expanding_garch_log, 'Expanding-window GARCH (DEXSZUS, USD/CHF)')


In [ ]:
# GARCH forecasts for 1 Oct 2026 to 12 Oct 2026, from the expanding-window cell
garch_oct = pd.DataFrame({
    'Forecast variance (%^2)': expanding_forecast_variances,
    'Forecast daily vol (%)': np.sqrt(expanding_forecast_variances),
    'Forecast vol (annualised %)': expanding_forecast_volatility,
    'Realised vol 30d (annualised %)': expanding_realized_volatility,
    'Actual return (%)': forecast_returns,
}).loc['2026-10-01':'2026-10-12']

print(garch_oct.round(4))

4. Fat tails and the expanding sample. If you fit GARCH with normal errors on FX returns, which have heavier tails than a normal distribution, crisis observations tend to inflate ω. With an expanding window, those crisis years never leave the sample. A Student‑t GARCH, or a rolling window of something like 5 years, would likely narrow the gap. That’s a nice robustness check to include in your write-up.

In [ ]:
# Rolling-window GARCH(1,1): 2005-2006 warm-up, followed by a maximum
# 5-year (1,260 trading-return) estimation window.

rolling_horizon = 21
rolling_window_size = 5 * 252

rolling_px = fx_df[['observation_date', 'DEXSZUS']].copy()
rolling_px['observation_date'] = pd.to_datetime(
    rolling_px['observation_date'], errors='coerce'
)
rolling_px['DEXSZUS'] = pd.to_numeric(
    rolling_px['DEXSZUS'], errors='coerce'
)
rolling_px = (
    rolling_px.dropna()
    .set_index('observation_date')
    .sort_index()
)

rolling_returns = (
    rolling_px['DEXSZUS'].pct_change(fill_method=None).dropna() * 100
)
rolling_initial = rolling_returns.loc['2005-01-01':'2006-12-31']
rolling_forecast = rolling_returns.loc['2007-01-01':'2026-12-31']

if rolling_initial.empty or rolling_forecast.empty:
    raise ValueError(
        'DEXSZUS data must include returns in 2005-2006 and 2007-2026.'
    )

rolling_history = rolling_initial.copy()
rolling_forecast_1d = pd.Series(
    index=rolling_forecast.index,
    dtype=float,
    name='One-day-ahead rolling-window GARCH variance'
)
rolling_forecast_21d = pd.Series(
    index=rolling_forecast.index,
    dtype=float,
    name='21-day average rolling-window GARCH variance'
)
rolling_garch_log = []

for step, (forecast_date, observed_return) in enumerate(
    rolling_forecast.items(), start=1
):
    rolling_model = arch_model(
        rolling_history,
        mean='Constant',
        vol='Garch',
        p=1,
        q=1,
        dist='normal'
    )
    rolling_fit = rolling_model.fit(disp='off')

    horizon_variances = rolling_fit.forecast(
        horizon=rolling_horizon,
        reindex=False
    ).variance.iloc[-1].values

    rolling_forecast_1d.loc[forecast_date] = horizon_variances[0]
    rolling_forecast_21d.loc[forecast_date] = horizon_variances.mean()

    rolling_params = rolling_fit.params
    rolling_alpha = rolling_params['alpha[1]']
    rolling_beta = rolling_params['beta[1]']
    rolling_persistence = rolling_alpha + rolling_beta

    rolling_garch_log.append({
        'date': forecast_date,
        'omega': rolling_params['omega'],
        'alpha': rolling_alpha,
        'beta': rolling_beta,
        'persistence': rolling_persistence,
        'halflife_days': (
            np.log(0.5) / np.log(rolling_persistence)
            if 0 < rolling_persistence < 1
            else np.nan
        )
    })

    rolling_history.loc[forecast_date] = observed_return
    rolling_history = rolling_history.iloc[-rolling_window_size:]

    if step % 500 == 0:
        print(
            f'Rolling GARCH forecast {step:,} of '
            f'{len(rolling_forecast):,} ({forecast_date.date()}); '
            f'estimation history capped at {rolling_window_size:,} returns.'
        )

rolling_garch_log = pd.DataFrame(rolling_garch_log).set_index('date')
rolling_forecast_volatility = (
    np.sqrt(rolling_forecast_1d) * np.sqrt(252)
)
rolling_realized_volatility = (
    rolling_returns.rolling(30, min_periods=30).std()
    * np.sqrt(252)
).reindex(rolling_forecast.index)

rolling_vix = vix_df[['observation_date', 'VIXCLS']].copy()
rolling_vix['observation_date'] = pd.to_datetime(
    rolling_vix['observation_date'], errors='coerce'
)
rolling_vix['VIXCLS'] = pd.to_numeric(
    rolling_vix['VIXCLS'], errors='coerce'
)
rolling_vix = (
    rolling_vix.dropna()
    .set_index('observation_date')['VIXCLS']
    .sort_index()
)
rolling_vix = rolling_vix.reindex(
    rolling_forecast.index, method='ffill'
)

rolling_regime = pd.Series(
    np.select(
        [rolling_vix > 30, rolling_vix > 15],
        ['High', 'Medium'],
        default='Low'
    ),
    index=rolling_vix.index
).where(rolling_vix.notna())

rolling_colors = {
    'Low': 'tab:green',
    'Medium': 'gold',
    'High': 'tab:red'
}

fig, ax = plt.subplots(figsize=(15, 6))
rolling_groups = rolling_regime.ne(
    rolling_regime.shift()
).cumsum()

for _, segment in rolling_regime.groupby(rolling_groups):
    regime = segment.iloc[0]
    if pd.notna(regime):
        ax.axvspan(
            segment.index[0],
            segment.index[-1] + pd.Timedelta(days=1),
            color=rolling_colors[regime],
            alpha=0.12,
            linewidth=0,
            zorder=0
        )

ax.plot(
    rolling_forecast_volatility.index,
    rolling_forecast_volatility,
    color='tab:blue',
    linewidth=1,
    label='Five-year rolling-window GARCH forecast'
)
ax.plot(
    rolling_realized_volatility.index,
    rolling_realized_volatility,
    color='tab:orange',
    linewidth=1,
    alpha=0.8,
    label='Trailing 30-day realized volatility'
)
ax.set_title(
    'DEXSZUS: Rolling-Window GARCH(1,1) '
    '(5-year estimation window)'
)
ax.set_xlabel('Forecast date')
ax.set_ylabel('Annualized volatility (%)')
ax.grid(True, linestyle=':', alpha=0.5)

handles, labels = ax.get_legend_handles_labels()
handles += [
    Patch(facecolor=rolling_colors[name], alpha=0.2,
          label=f'{name} VIX regime')
    for name in ['Low', 'Medium', 'High']
]
labels += [name for name in [
    'Low VIX regime',
    'Medium VIX regime',
    'High VIX regime'
]]
ax.legend(handles=handles)
plt.tight_layout()
plt.show()

print(
    f'Initial warm-up: '
    f'{rolling_initial.index.min().date()} to '
    f'{rolling_initial.index.max().date()}'
)
print(
    f'Forecast period: '
    f'{rolling_forecast.index.min().date()} to '
    f'{rolling_forecast.index.max().date()}'
)
print(
    f'Rolling refits: '
    f'{rolling_forecast_volatility.notna().sum():,}; '
    f'estimation window: {rolling_window_size:,} observations'
)

In [ ]:
# ===== GARCH(1,1) parameters: 5-year rolling window =====
show_garch_params(rolling_garch_log, '5-year rolling-window GARCH (DEXSZUS, USD/CHF)')


In [ ]:
jpy_oos_garch_alpha_beta_annual = (
    expanding_garch_log
    .assign(year=expanding_garch_log.index.year)
    .groupby('year')[['alpha', 'beta', 'persistence']]
    .median()
)

print('Median GARCH alphas and betas by year:')
print(jpy_oos_garch_alpha_beta_annual.round(6).to_string())

Realized variance from intraday data is the gold standard. You sum squared 5-minute returns over the day: RV = Σ r²ᵢ. Sampling every 5 minutes is the usual compromise, because finer sampling picks up bid-ask “microstructure noise.” The forexsb.com data you’ve been using has intraday USDJPY bars, so you can compute this already.

In [ ]:
# Two-regime Markov Regime Switching model on USD/CHF daily % returns (DEXSZUS)
# Methodology follows Öztürk's MRS tutorial: stationarity tests -> AR order by AIC -> 2-regime
# model with switching variance -> diagnostics -> variance / transition / regime plots.

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
from statsmodels.tsa.regime_switching.markov_autoregression import MarkovAutoregression
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import normaltest

warnings.filterwarnings('ignore', category=UserWarning)   # silences statsmodels date-index / KPSS warnings

# ---------- 1. Daily percentage returns of DEXSZUS (same cleaning as the earlier cells) ----------
jpy_mrs_px = fx_df[['observation_date', 'DEXSZUS']].copy()
jpy_mrs_px['observation_date'] = pd.to_datetime(jpy_mrs_px['observation_date'], errors='coerce')
jpy_mrs_px['DEXSZUS'] = pd.to_numeric(jpy_mrs_px['DEXSZUS'], errors='coerce')
jpy_mrs_px = jpy_mrs_px.dropna().set_index('observation_date').sort_index()

jpy_mrs_data = pd.DataFrame({'Returns': jpy_mrs_px['DEXSZUS'].pct_change(fill_method=None) * 100}).dropna()
print(f"Observations: {len(jpy_mrs_data):,} ({jpy_mrs_data.index.min().date()} to {jpy_mrs_data.index.max().date()})")

# ---------- 2. Stationarity tests (ADF + KPSS) ----------
adf_stat, adf_p = adfuller(jpy_mrs_data['Returns'])[:2]
kpss_stat, kpss_p = kpss(jpy_mrs_data['Returns'], nlags='auto')[:2]
print("\n--- Stationarity Tests ---")
print(f"ADF  statistic: {adf_stat:.4f} | p-value: {adf_p:.4f} -> {'stationary' if adf_p < 0.05 else 'likely non-stationary'}")
print(f"KPSS statistic: {kpss_stat:.4f} | p-value: {kpss_p:.4f} -> {'likely non-stationary' if kpss_p < 0.05 else 'stationary'}")

# ---------- 3. Best AR order (0-5) by AIC, always with 2 regimes ----------
# Note: the tutorial passes `order` to MarkovRegression, but in statsmodels that argument is not an AR lag.
# Order 0 = MarkovRegression (regime-dependent mean + variance); order p > 0 = MarkovAutoregression (AR(p)).
def jpy_mrs_fit(order, **fit_kwargs):
    if order == 0:
        mod = MarkovRegression(jpy_mrs_data['Returns'], k_regimes=2, switching_variance=True)
    else:
        mod = MarkovAutoregression(jpy_mrs_data['Returns'], k_regimes=2, order=order,
                                   switching_ar=False, switching_variance=True)
    return mod.fit(**fit_kwargs)

jpy_mrs_max_order = 5
jpy_mrs_best_aic, jpy_mrs_best_order = np.inf, 0
for order in range(jpy_mrs_max_order + 1):
    try:
        trial = jpy_mrs_fit(order, disp=False)
        print(f"AR order {order}: AIC = {trial.aic:.2f}")
        if trial.aic < jpy_mrs_best_aic:
            jpy_mrs_best_aic, jpy_mrs_best_order = trial.aic, order
    except Exception as e:
        print(f"AR order {order}: failed ({type(e).__name__})")
print(f"\nBest AR order (by AIC): {jpy_mrs_best_order}")

# ---------- 4. Fit the final two-regime Markov switching model ----------
jpy_mrs_res = jpy_mrs_fit(jpy_mrs_best_order)
print(jpy_mrs_res.summary())

# Label regimes by estimated variance so "Calm" / "Turbulent" are unambiguous
jpy_mrs_var = pd.Series({i: jpy_mrs_res.params[f'sigma2[{i}]'] for i in range(2)})
calm, turb = jpy_mrs_var.idxmin(), jpy_mrs_var.idxmax()
jpy_mrs_names = {calm: 'Low-vol (calm)', turb: 'High-vol (turbulent)'}

# ---------- 5. Diagnostics ----------
print("\n--- Model Diagnostics ---")
try:
    jpy_mrs_resid = jpy_mrs_res.resid
except AttributeError:   # fallback: standardised residual from the smoothed-probability mixture (order 0 only)
    _p = jpy_mrs_res.smoothed_marginal_probabilities
    _mu = sum(_p[i] * jpy_mrs_res.params[f'const[{i}]'] for i in range(2))
    _sd = np.sqrt(sum(_p[i] * jpy_mrs_res.params[f'sigma2[{i}]'] for i in range(2)))
    jpy_mrs_resid = (jpy_mrs_data['Returns'].loc[_p.index] - _mu) / _sd
print("Ljung-Box (lag 10) on residuals:")
print(acorr_ljungbox(jpy_mrs_resid, lags=[10], return_df=True))
norm = normaltest(jpy_mrs_resid)
print(f"\nD'Agostino-Pearson normality: stat = {norm.statistic:.4f}, p = {norm.pvalue:.4f} -> "
      f"{'residuals not normal' if norm.pvalue < 0.05 else 'residuals look normal'}")

# ---------- 6. Key parameters ----------
print("\n--- Key Parameters ---")
print(f"AIC: {jpy_mrs_res.aic:.2f} | BIC: {jpy_mrs_res.bic:.2f} | Log-Likelihood: {jpy_mrs_res.llf:.2f}")
for i in (calm, turb):
    print(f"{jpy_mrs_names[i]}: variance = {jpy_mrs_var[i]:.4f}, daily vol = {np.sqrt(jpy_mrs_var[i]):.3f}%, "
          f"annualised vol = {np.sqrt(jpy_mrs_var[i] * 252):.1f}%, mean = {jpy_mrs_res.params[f'const[{i}]']:.4f}%")

# Transition matrix: statsmodels stores P(to i | from j) in [i, j]; transpose so rows = current, cols = next
jpy_mrs_P = np.asarray(jpy_mrs_res.regime_transition)
if jpy_mrs_P.ndim == 3:
    jpy_mrs_P = jpy_mrs_P[:, :, 0]
jpy_mrs_P = jpy_mrs_P.T
print("\nTransition matrix (rows = current regime, cols = next regime):")
print(pd.DataFrame(jpy_mrs_P, index=[jpy_mrs_names[i] for i in range(2)],
                   columns=[jpy_mrs_names[i] for i in range(2)]).round(4))
print("\nExpected regime durations (days):")
for i in (calm, turb):
    print(f"  {jpy_mrs_names[i]}: {1 / (1 - jpy_mrs_P[i, i]):.1f}")

# ---------- 7. Plots ----------
probs = jpy_mrs_res.smoothed_marginal_probabilities
jpy_mrs_regime = probs.idxmax(axis=1)
jpy_mrs_ret = jpy_mrs_data['Returns'].loc[probs.index]      # aligned (AR terms drop the first `order` obs)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(jpy_mrs_data.index, jpy_mrs_data['Returns'], color='steelblue', linewidth=0.7)
ax.set_title('DEXSZUS: Daily Percentage Returns')
ax.set_ylabel('Return (%)'); ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar([jpy_mrs_names[i] for i in (calm, turb)], [jpy_mrs_var[calm], jpy_mrs_var[turb]],
            color=['tab:green', 'tab:red'], alpha=0.8)
axes[0].set_title('Estimated Variance by Regime'); axes[0].set_ylabel('Variance')
sns.heatmap(jpy_mrs_P, annot=True, fmt='.4f', cmap='Reds', ax=axes[1],
            xticklabels=[jpy_mrs_names[i] for i in range(2)],
            yticklabels=[jpy_mrs_names[i] for i in range(2)])
axes[1].set_title('Regime Transition Matrix'); axes[1].set_xlabel('Next regime'); axes[1].set_ylabel('Current regime')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
for i, colour in ((calm, 'tab:green'), (turb, 'tab:red')):
    mask = jpy_mrs_regime == i
    ax.scatter(jpy_mrs_ret.index[mask], jpy_mrs_ret[mask], s=6, alpha=0.6, color=colour, label=jpy_mrs_names[i])
ax.set_title('DEXSZUS: Regime Assignment (most likely smoothed regime)')
ax.set_ylabel('Return (%)'); ax.legend(); ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
for i, colour in ((calm, 'tab:green'), (turb, 'tab:red')):
    ax.plot(probs[i], color=colour, linewidth=0.9, label=f'P({jpy_mrs_names[i]})')
ax.set_title('DEXSZUS: Smoothed Regime Probabilities Over Time')
ax.set_ylim(-0.02, 1.02); ax.legend(); ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout(); plt.show()


In [ ]:
jpy_now_px = fx_df[['observation_date', 'DEXSZUS']].copy()
jpy_now_px['observation_date'] = pd.to_datetime(jpy_now_px['observation_date'], errors='coerce')
jpy_now_px['DEXSZUS'] = pd.to_numeric(jpy_now_px['DEXSZUS'], errors='coerce')
jpy_now_px = jpy_now_px.dropna().set_index('observation_date').sort_index()
jpy_now_returns = jpy_now_px['DEXSZUS'].pct_change(fill_method=None).dropna() * 100

jpy_now_last = jpy_now_returns.index[-1]
jpy_now_target = jpy_now_last + pd.offsets.BDay(1)     # next weekday after the last observation
print(f"Last observation in the data: {jpy_now_last.date()} "
      f"(return {jpy_now_returns.iloc[-1]:+.3f}%). Forecasting for {jpy_now_target.date()}.")
print(f"Today's date is {pd.Timestamp.today().date()}; if the data end earlier, the data source is lagging "
      f"and 'now' means the latest available date.\n")

# ---------- Fit on the full sample ----------
jpy_now_mod = MarkovRegression(jpy_now_returns, k_regimes=2, switching_variance=True)
jpy_now_res = jpy_now_mod.fit(disp=False)

jpy_now_p = pd.Series(np.asarray(jpy_now_res.params), index=jpy_now_mod.param_names)
mu = np.array([jpy_now_p['const[0]'], jpy_now_p['const[1]']])
s2 = np.array([jpy_now_p['sigma2[0]'], jpy_now_p['sigma2[1]']])
calm, turb = int(np.argmin(s2)), int(np.argmax(s2))
names = {calm: 'Calm (low variance)', turb: 'Turbulent (high variance)'}

P = np.asarray(jpy_now_res.regime_transition)[:, :, 0]          # P[i, j] = Prob(to i | from j)
filtered = np.asarray(jpy_now_res.filtered_marginal_probabilities)   # uses only data up to each date
filtered_now = filtered[-1]

print("--- Fitted regimes ---")
for i in (calm, turb):
    print(f"{names[i]}: mean = {mu[i]:+.4f}%/day, daily vol = {np.sqrt(s2[i]):.3f}%, "
          f"annualised vol = {np.sqrt(s2[i] * 252):.1f}%, expected duration = {1 / (1 - P[i, i]):.1f} days")

# ---------- Current regime (nowcast) ----------
print(f"\n--- Current regime as of {jpy_now_last.date()} (filtered probabilities) ---")
for i in (calm, turb):
    print(f"P({names[i]}) = {filtered_now[i]:.1%}")
print(f"Most likely regime: {names[int(np.argmax(filtered_now))]}")

# ---------- Forecasts: 1 to 21 trading days ahead ----------
def jpy_now_mixture_var(probs):
    return probs @ (s2 + mu ** 2) - (probs @ mu) ** 2

jpy_now_horizons = np.arange(1, 22)
jpy_now_fc = pd.DataFrame(index=jpy_now_horizons, columns=['P(turbulent)', 'Forecast variance', 'Forecast vol (annualised %)'], dtype=float)
for h in jpy_now_horizons:
    probs_h = np.linalg.matrix_power(P, int(h)) @ filtered_now
    jpy_now_fc.loc[h, 'P(turbulent)'] = probs_h[turb]
    jpy_now_fc.loc[h, 'Forecast variance'] = jpy_now_mixture_var(probs_h)
    jpy_now_fc.loc[h, 'Forecast vol (annualised %)'] = np.sqrt(jpy_now_fc.loc[h, 'Forecast variance'] * 252)
jpy_now_fc.index.name = 'Days ahead'

jpy_now_h1 = jpy_now_fc.loc[1]
jpy_now_realized = jpy_now_returns.rolling(30).std().iloc[-1] * np.sqrt(252)
print(f"\n--- Forecast for {jpy_now_target.date()} (next trading day) ---")
print(f"P(turbulent regime): {jpy_now_h1['P(turbulent)']:.1%}")
print(f"Forecast daily variance: {jpy_now_h1['Forecast variance']:.4f} (%^2)  ->  daily vol {np.sqrt(jpy_now_h1['Forecast variance']):.3f}%")
print(f"Forecast annualised volatility: {jpy_now_h1['Forecast vol (annualised %)']:.2f}%")
print(f"For reference, trailing 30-day realised volatility the previous trading day: {jpy_now_realized:.2f}% (annualised)")
print(f"Average annualised vol over the next 21 trading days: "
      f"{np.sqrt(jpy_now_fc['Forecast variance'].mean() * 252):.2f}%")
print("\nForecast path:")
print(jpy_now_fc.loc[[1, 2, 5, 10, 21]].round(4))

# ---------- Plots ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
recent = pd.Series(filtered[:, turb], index=jpy_now_returns.index).iloc[-250:]
axes[0].plot(recent.index, recent, color='tab:red', linewidth=1)
axes[0].set_ylim(-0.02, 1.02); axes[0].grid(True, linestyle=':', alpha=0.5)
axes[0].set_title('Filtered P(turbulent regime), last 250 trading days'); axes[0].set_ylabel('Probability')

"""Here, “filtered” describes how the model calculates the regime probabilities: 
for each date t, it uses observations up through t to estimate the regime probability 
at that date, given the model’s parameters. It does not mean the parameters were 
estimated using only data available up to t."""

axes[1].plot(jpy_now_fc.index, jpy_now_fc['Forecast vol (annualised %)'], marker='o', markersize=3, color='tab:purple', label='Forecast')
axes[1].axhline(np.sqrt(s2[calm] * 252), color='tab:green', linestyle='--', label='Calm regime vol')
axes[1].axhline(np.sqrt(s2[turb] * 252), color='tab:red', linestyle='--', label='Turbulent regime vol')
axes[1].axhline(jpy_now_realized, color='tab:orange', linestyle=':', label='Trailing 30d realised')
axes[1].set_title('Volatility forecast by horizon'); axes[1].set_xlabel('Trading days ahead')
axes[1].set_ylabel('Annualised volatility (%)'); axes[1].grid(True, linestyle=':', alpha=0.5); axes[1].legend()
plt.tight_layout(); plt.show()

filtered means the model estimates the probability of each volatility regime using only data available up to that date.

## Diagnostics: expanding-window Markov switching vs GARCH (regime parameters, α+β, 21-day forecasts)

The cells below reuse your conventions: `fx_df` / `vix_df`, `pct_change` returns in %, the 2005-2006 initial fit,
daily refits from 2007, `jpy_oos_*` names for the Markov loop and `expanding_*` names for GARCH.

* Cell A (setup) and Cell B **replace** your original expanding-window Markov cell. Cell B now contains that cell's loop,
  evaluation table and plots together with the parameter log and 21-day forecasts, so the loop only runs once and the
  original cell has been removed. It still produces `jpy_oos_var`, `jpy_oos_phigh`, `jpy_oos_fc_vol`, ...
* Cell C is the GARCH counterpart: it produces the same `expanding_forecast_variances` as your original GARCH cell plus extra outputs,
  so you do not need to run both of those.
* **Date convention (yours):** the forecast stored at date D uses data through D-1. A 21-day forecast at D therefore covers
  D ... D+20, and the realised target is the mean of squared returns over D ... D+20.

In [ ]:
# ===== Cell A: setup, parameters, data, targets =====
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import statsmodels.api as sm
from arch import arch_model
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

warnings.filterwarnings('ignore')
np.random.seed(0)

# ---------- Parameters ----------
jpy_oos_refit_every = 1      # as in your cell: 1 = refit the Markov model every day (parameters frozen between refits otherwise)
jpy_oos_k = 2                # number of Markov regimes; try 3 as a robustness check
jpy_oos_cold_every = 250     # every n-th refit ALSO tries random restarts and keeps the better likelihood (0 = never)
jpy_oos_H = 21               # multi-day horizon (trading days)
jpy_oos_run_sim = True       # run the two-regime simulation at the end

# ---------- Data: identical cleaning and split to your expanding GARCH / Markov cells ----------
jpy_oos_px = fx_df[['observation_date', 'DEXSZUS']].copy()
jpy_oos_px['observation_date'] = pd.to_datetime(jpy_oos_px['observation_date'], errors='coerce')
jpy_oos_px['DEXSZUS'] = pd.to_numeric(jpy_oos_px['DEXSZUS'], errors='coerce')
jpy_oos_px = jpy_oos_px.dropna().set_index('observation_date').sort_index()
jpy_oos_returns = jpy_oos_px['DEXSZUS'].pct_change(fill_method=None).dropna() * 100

jpy_oos_initial = jpy_oos_returns.loc['2005-01-01':'2006-12-31']
jpy_oos_forecast = jpy_oos_returns.loc['2007-01-01':'2026-12-31']
if jpy_oos_initial.empty or jpy_oos_forecast.empty:
    raise ValueError('DEXSZUS data must include returns in both 2005-2006 and 2007-2026.')
jpy_oos_idx = jpy_oos_forecast.index

# ---------- Realised targets, dated like the forecasts (a forecast at D uses data through D-1) ----------
jpy_oos_r2 = jpy_oos_returns ** 2
jpy_oos_y1 = jpy_oos_r2.reindex(jpy_oos_idx)                                                    # r^2 on day D
jpy_oos_yH = jpy_oos_r2.rolling(jpy_oos_H).mean().shift(-(jpy_oos_H - 1)).reindex(jpy_oos_idx)   # mean r^2 over D ... D+H-1
jpy_oos_naive_var = (jpy_oos_returns.rolling(window=30, min_periods=30).var().shift(1)
                     .reindex(jpy_oos_idx))                                                      # your trailing-30d benchmark
jpy_oos_realized = (jpy_oos_returns.rolling(window=30, min_periods=30).std() * np.sqrt(252)).reindex(jpy_oos_idx)

# ---------- VIX regimes (your thresholds: >30 High, >15 Medium, else Low) ----------
jpy_oos_vix = vix_df[['observation_date', 'VIXCLS']].copy()
jpy_oos_vix['observation_date'] = pd.to_datetime(jpy_oos_vix['observation_date'], errors='coerce')
jpy_oos_vix['VIXCLS'] = pd.to_numeric(jpy_oos_vix['VIXCLS'], errors='coerce')
jpy_oos_vix = jpy_oos_vix.dropna().set_index('observation_date')['VIXCLS'].sort_index()
jpy_oos_vix = jpy_oos_vix.reindex(jpy_oos_idx, method='ffill')
jpy_oos_vix_regime = pd.Series(np.select([jpy_oos_vix > 30, jpy_oos_vix > 15], ['High', 'Medium'], default='Low'),
                               index=jpy_oos_vix.index).where(jpy_oos_vix.notna())
jpy_oos_colors = {'Low': 'tab:green', 'Medium': 'gold', 'High': 'tab:red'}

def jpy_oos_shade(ax):
    groups = jpy_oos_vix_regime.ne(jpy_oos_vix_regime.shift()).cumsum()
    for _, seg in jpy_oos_vix_regime.groupby(groups):
        if pd.notna(seg.iloc[0]):
            ax.axvspan(seg.index[0], seg.index[-1] + pd.Timedelta(days=1), color=jpy_oos_colors[seg.iloc[0]],
                       alpha=0.12, linewidth=0, zorder=0)

print(f'Initial fit: {jpy_oos_initial.index.min().date()} to {jpy_oos_initial.index.max().date()} | '
      f'forecasts: {jpy_oos_idx.min().date()} to {jpy_oos_idx.max().date()} ({len(jpy_oos_idx):,} dates)')

In [ ]:
# ===== Cell B (merged): expanding-window Markov loop + parameter log + 21-day forecasts + evaluation + plots =====
# This cell replaces your original expanding-window Markov cell: same loop, so it only needs to run once.
# It relies on Cell A for the data, targets, VIX regimes and parameters (jpy_oos_returns, jpy_oos_naive_var, jpy_oos_realized,
# jpy_oos_vix_regime, jpy_oos_shade, jpy_oos_colors, ...).
# Same spec as your cell: MarkovRegression(history, switching mean and variance), warm start, one-day-ahead variance
#   Var = sum_i q_i (sigma2_i + mu_i^2) - (sum_i q_i mu_i)^2 ,  q = P @ filtered_probs(yesterday).
# Added: the same quantity averaged over the next H days (q_h = P^h @ filtered_probs), and a log of the parameters.
k, H = jpy_oos_k, jpy_oos_H
jpy_oos_history = jpy_oos_initial.copy()
jpy_oos_var = pd.Series(index=jpy_oos_idx, dtype=float, name='MRS one-day-ahead forecast variance')
jpy_oos_var_H = pd.Series(index=jpy_oos_idx, dtype=float, name=f'MRS {H}-day average forecast variance')
jpy_oos_phigh = pd.Series(index=jpy_oos_idx, dtype=float, name='P(high-variance regime tomorrow)')
jpy_oos_log_rows = []
jpy_oos_params, jpy_oos_res, jpy_oos_failures, jpy_oos_n_refits = None, None, 0, 0
jpy_oos_start = time.time()

for step, (forecast_date, observed_return) in enumerate(jpy_oos_forecast.items(), start=1):
    mod = MarkovRegression(jpy_oos_history, k_regimes=k, switching_variance=True)
    do_refit = jpy_oos_res is None or (step - 1) % jpy_oos_refit_every == 0
    refit_done = False
    try:
        if do_refit:
            jpy_oos_n_refits += 1
            cold = jpy_oos_params is None or (jpy_oos_cold_every and jpy_oos_n_refits % jpy_oos_cold_every == 0)
            fits = []
            if jpy_oos_params is not None:                      # warm start from the previous estimates
                try:
                    fits.append(mod.fit(start_params=jpy_oos_params, disp=False))
                except Exception:
                    pass
            if cold or not fits:                                # random restarts guard against local optima
                fits.append(mod.fit(search_reps=10, disp=False))
            jpy_oos_res = max(fits, key=lambda r: r.llf)
            jpy_oos_params = np.asarray(jpy_oos_res.params)
            refit_done = True
        else:
            jpy_oos_res = mod.smooth(jpy_oos_params)            # keep parameters, just update the filter
    except Exception:
        if jpy_oos_params is None:
            raise
        jpy_oos_failures += 1                                   # fall back to the last good parameters
        jpy_oos_res = mod.smooth(jpy_oos_params)

    p = pd.Series(jpy_oos_params, index=mod.param_names)
    mu = np.array([p[f'const[{i}]'] for i in range(k)])
    s2 = np.array([p[f'sigma2[{i}]'] for i in range(k)])
    P = np.asarray(jpy_oos_res.regime_transition)[:, :, 0]      # P[i, j] = Prob(to i | from j)
    f_today = np.asarray(jpy_oos_res.filtered_marginal_probabilities)[-1]

    q = P @ f_today                                             # regime probabilities for the forecast date
    jpy_oos_var.loc[forecast_date] = q @ (s2 + mu ** 2) - (q @ mu) ** 2
    jpy_oos_phigh.loc[forecast_date] = q[np.argmax(s2)]

    q_h, acc = f_today, 0.0                                     # average expected variance over D ... D+H-1
    for _ in range(H):
        q_h = P @ q_h
        acc += q_h @ (s2 + mu ** 2) - (q_h @ mu) ** 2
    jpy_oos_var_H.loc[forecast_date] = acc / H

    if refit_done:                                              # parameter log, regimes ordered calm -> turbulent
        order = np.argsort(s2)
        stay = np.diag(P)[order]
        row = {'date': forecast_date, 'llf': jpy_oos_res.llf,
               'decay': sorted(np.abs(np.linalg.eigvals(P)))[-2]}   # 2nd eigenvalue: equals p00+p11-1 when k=2
        for r_, i in enumerate(order):
            row[f'sigma_ann_{r_}'] = np.sqrt(s2[i] * 252)
            row[f'mean_{r_}'] = mu[i]
            row[f'stay_{r_}'] = stay[r_]
            row[f'exp_days_{r_}'] = 1 / (1 - stay[r_]) if stay[r_] < 1 else np.inf
        jpy_oos_log_rows.append(row)

    jpy_oos_history.loc[forecast_date] = observed_return        # add today's return only after its forecast is recorded

    if step % 250 == 0:
        print(f'Forecast {step:,} of {len(jpy_oos_forecast):,} ({forecast_date.date()}), '
              f'{(time.time() - jpy_oos_start) / 60:.1f} min elapsed, {jpy_oos_failures} fit failures.')

jpy_oos_log = pd.DataFrame(jpy_oos_log_rows).set_index('date')
jpy_oos_fc_vol = np.sqrt(jpy_oos_var) * np.sqrt(252)
print(f'Done: {len(jpy_oos_log):,} refits logged.')


# ---------- Forecast evaluation (the table from your original Markov cell: 1-day-ahead forecasts) ----------
# Benchmark: trailing 30-day variance known the day before (a "no-model" forecast), defined in Cell A.
jpy_oos_candidates = {'Markov switching (expanding)': jpy_oos_var, 'Trailing 30d variance (naive)': jpy_oos_naive_var}
if 'expanding_forecast_variances' in globals():      # GARCH results from the earlier GARCH cell (or Cell C), if it has been run
    jpy_oos_candidates['GARCH(1,1) (expanding)'] = expanding_forecast_variances.reindex(jpy_oos_forecast.index)

# Evaluate every model on the same dates
jpy_oos_common = pd.concat(jpy_oos_candidates, axis=1).dropna().index
jpy_oos_sq = jpy_oos_forecast.loc[jpy_oos_common] ** 2        # squared return = noisy proxy for true daily variance
jpy_oos_rv = jpy_oos_realized.loc[jpy_oos_common]

rows = {}
for name, v in jpy_oos_candidates.items():
    v = v.loc[jpy_oos_common]
    vol = np.sqrt(v * 252)
    rows[name] = {
        'MSE vs r^2': ((jpy_oos_sq - v) ** 2).mean(),
        'QLIKE vs r^2': (np.log(v) + jpy_oos_sq / v).mean(),
        'RMSE vs 30d realized vol (%)': np.sqrt(((vol - jpy_oos_rv) ** 2).mean()),
        'MAE vs 30d realized vol (%)': (vol - jpy_oos_rv).abs().mean(),
        'Corr with 30d realized vol': vol.corr(jpy_oos_rv),
    }
print(f'\nEvaluation on {len(jpy_oos_common):,} common forecast dates '
      f'({jpy_oos_common.min().date()} to {jpy_oos_common.max().date()}); lower is better except correlation.')
print(pd.DataFrame(rows).T.round(4))

# ---------- Plots ----------
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(15, 9), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
jpy_oos_shade(ax)                                    # VIX-regime shading, defined in Cell A
jpy_oos_shade(ax2)
ax.plot(jpy_oos_fc_vol.index, jpy_oos_fc_vol, color='tab:purple', linewidth=1,
        label='Daily-refit expanding-window Markov-switching forecast (annualized)')
if 'expanding_forecast_variances' in globals():
    garch_vol = np.sqrt(expanding_forecast_variances.reindex(jpy_oos_forecast.index)) * np.sqrt(252)
    ax.plot(garch_vol.index, garch_vol, color='tab:blue', linewidth=1, alpha=0.7, label='Expanding-window GARCH forecast (annualized)')
ax.plot(jpy_oos_realized.index, jpy_oos_realized, color='tab:orange', linewidth=1, alpha=0.8,
        label='Trailing 30-day realized volatility (annualized)')
ax.set_title('DEXSZUS: Expanding-Window Real-Time Markov-Switching Volatility Forecasts (2007-2026)')
ax.set_ylabel('Annualized volatility (%)'); ax.grid(True, linestyle=':', alpha=0.5)
handles, labels = ax.get_legend_handles_labels()
patches = [Patch(facecolor=jpy_oos_colors[n], alpha=0.2, label=f'{n} VIX regime') for n in ['Low', 'Medium', 'High']]
ax.legend(handles=handles + patches, labels=labels + [h.get_label() for h in patches])

ax2.plot(jpy_oos_phigh.index, jpy_oos_phigh, color='tab:red', linewidth=0.9)
ax2.set_ylim(-0.02, 1.02); ax2.set_ylabel('P(high-variance regime)')
ax2.set_xlabel('Forecast date'); ax2.grid(True, linestyle=':', alpha=0.5)
ax2.set_title('Forecast probability of tomorrow being in the high-variance regime')
fig.tight_layout(); plt.show()

print(f'Initial fit period: {jpy_oos_initial.index.min().date()} to {jpy_oos_initial.index.max().date()}')
print(f'Forecast period: {jpy_oos_forecast.index.min().date()} to {jpy_oos_forecast.index.max().date()}')
print(f'Daily forecasts: {jpy_oos_var.notna().sum():,} | fits that failed and reused prior parameters: {jpy_oos_failures}')

In [ ]:
# Daily volatility forecasts from the out-of-sample start through the next weekday
# after the latest observation. Volatility values are daily percentages.

# Existing expanding-window forecasts are dated on the day being forecast.
forecast_table = pd.DataFrame({
    'Markov forecast vol (%)': np.sqrt(jpy_oos_var),
    'GARCH forecast vol (%)': np.sqrt(expanding_forecast_variances),
    'Realized vol (%)': jpy_oos_returns.abs()
}).reindex(jpy_oos_idx)

# Refit using all returns through the latest observed date to forecast the next weekday.
last_date = jpy_oos_returns.index.max()
next_date = last_date + pd.offsets.BDay(1)

# Final Markov fit and one-step-ahead variance forecast
final_mrs_mod = MarkovRegression(
    jpy_oos_returns, k_regimes=jpy_oos_k, switching_variance=True
)
try:
    final_mrs_res = final_mrs_mod.fit(
        start_params=np.asarray(jpy_oos_params), disp=False
    )
except Exception:
    final_mrs_res = final_mrs_mod.fit(disp=False)

final_mrs_params = pd.Series(
    np.asarray(final_mrs_res.params), index=final_mrs_mod.param_names
)
final_mu = np.array([final_mrs_params[f'const[{i}]'] for i in range(jpy_oos_k)])
final_s2 = np.array([final_mrs_params[f'sigma2[{i}]'] for i in range(jpy_oos_k)])

final_P = np.asarray(final_mrs_res.regime_transition)
if final_P.ndim == 3:
    final_P = final_P[:, :, -1]
final_filtered = np.asarray(final_mrs_res.filtered_marginal_probabilities)[-1]
next_regime_probs = final_P @ final_filtered
final_mrs_variance = (
    next_regime_probs @ (final_s2 + final_mu**2)
    - (next_regime_probs @ final_mu)**2
)

# Final GARCH fit and one-step-ahead variance forecast
final_garch_fit = arch_model(
    jpy_oos_returns, mean='Constant', vol='Garch', p=1, q=1, dist='normal'
).fit(disp='off')
final_garch_variance = final_garch_fit.forecast(
    horizon=1, reindex=False
).variance.iloc[-1, 0]

forecast_table.loc[next_date] = [
    np.sqrt(final_mrs_variance),
    np.sqrt(final_garch_variance),
    np.nan
]
forecast_table.index.name = 'Forecast date'

print(f'Forecasts use data through {last_date.date()}; next weekday: {next_date.date()}')
print('Realized volatility is |daily return| (%); it is unavailable for the final forecast date.')
print(forecast_table.to_string(float_format=lambda x: f'{x:.4f}'))

In [ ]:
# ===== Cell C: expanding-window GARCH loop (your loop + alpha/beta log + 21-day forecasts) =====
# Same spec as your cell: arch_model(history, Constant mean, GARCH(1,1), normal), refit every day.
# Added: horizon-H forecast (average variance over D ... D+H-1) and a log of omega, alpha, beta, alpha+beta.
H = jpy_oos_H
expanding_history = jpy_oos_initial.copy()
expanding_forecast_variances = pd.Series(index=jpy_oos_idx, dtype=float, name='One-day-ahead GARCH forecast variance')
expanding_forecast_variances_H = pd.Series(index=jpy_oos_idx, dtype=float, name=f'GARCH {H}-day average forecast variance')
expanding_garch_rows = []
garch_start = time.time()

for step, (forecast_date, observed_return) in enumerate(jpy_oos_forecast.items(), start=1):
    expanding_model = arch_model(expanding_history, mean='Constant', vol='Garch', p=1, q=1, dist='normal')
    expanding_fit = expanding_model.fit(disp='off')
    fv = expanding_fit.forecast(horizon=H, reindex=False).variance.iloc[-1].values   # variances for D ... D+H-1
    expanding_forecast_variances.loc[forecast_date] = fv[0]
    expanding_forecast_variances_H.loc[forecast_date] = fv.mean()

    pr = expanding_fit.params
    om, al, be = pr['omega'], pr['alpha[1]'], pr['beta[1]']
    pers = al + be
    expanding_garch_rows.append({
        'date': forecast_date, 'omega': om, 'alpha': al, 'beta': be, 'persistence': pers,
        'halflife_days': np.log(0.5) / np.log(pers) if 0 < pers < 1 else np.nan,
        'longrun_vol_ann': np.sqrt(om / (1 - pers) * 252) if pers < 1 else np.nan})

    expanding_history.loc[forecast_date] = observed_return      # add today's return only after its forecast is recorded
    if step % 500 == 0:
        print(f'Refitted through forecast {step:,} of {len(jpy_oos_forecast):,} ({forecast_date.date()}), '
              f'{(time.time() - garch_start) / 60:.1f} min elapsed.')

expanding_garch_log = pd.DataFrame(expanding_garch_rows).set_index('date')
expanding_forecast_volatility = np.sqrt(expanding_forecast_variances) * np.sqrt(252)
expanding_realized_volatility = jpy_oos_realized
print('Done.')

### Parameter diagnostics
Regime volatilities, stay probabilities and expected durations from the Markov refits, next to the GARCH α, β and α+β.
Compare α+β with the Markov decay rate (the second eigenvalue of the transition matrix): both measure how fast a shock fades.

In [ ]:
# ===== Cell D: parameter diagnostics =====
k = jpy_oos_k
combined_log = jpy_oos_log.join(expanding_garch_log.add_prefix('g_'), how='outer')
cols = ([f'sigma_ann_{i}' for i in range(k)] + [f'stay_{i}' for i in range(k)] + [f'exp_days_{i}' for i in range(k)]
        + ['decay', 'g_alpha', 'g_beta', 'g_persistence', 'g_halflife_days', 'g_longrun_vol_ann'])
print('=== Yearly medians of the refit parameters ===')
print(combined_log[cols].groupby(combined_log.index.year).median().round(3).to_string())

fig, ax = plt.subplots(2, 2, figsize=(15, 8), sharex=True)
for i in range(k):
    ax[0, 0].plot(jpy_oos_log.index, jpy_oos_log[f'sigma_ann_{i}'], label=f'regime {i} ' + ('(calm)' if i == 0 else '(highest variance)' if i == k - 1 else ''))
    ax[0, 1].plot(jpy_oos_log.index, jpy_oos_log[f'exp_days_{i}'], label=f'regime {i}')
ax[0, 0].set_title('Markov: regime volatility at each refit (annualised %)'); ax[0, 0].legend()
ax[0, 1].set_title('Markov: model-implied expected regime duration (days, log scale)'); ax[0, 1].set_yscale('log'); ax[0, 1].legend()
ax[1, 0].plot(expanding_garch_log.index, expanding_garch_log['persistence'], label='GARCH alpha+beta')
ax[1, 0].plot(jpy_oos_log.index, jpy_oos_log['decay'], label='Markov decay rate (2nd eigenvalue of P)')
ax[1, 0].set_title('Persistence: GARCH alpha+beta vs Markov decay rate'); ax[1, 0].legend()
ax[1, 1].plot(expanding_garch_log.index, expanding_garch_log['alpha'], label='alpha')
ax[1, 1].plot(expanding_garch_log.index, expanding_garch_log['beta'], label='beta')
ax[1, 1].set_title('GARCH: alpha and beta'); ax[1, 1].legend()
for a in ax.ravel():
    a.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout(); plt.show()

In [ ]:
# ===== Cell E: flicker check on the high-variance regime =====
# Does the high-variance regime last for weeks, or flip on single large returns?
ph = jpy_oos_phigh.dropna()
flag = ph > 0.5
runs = (flag != flag.shift()).cumsum()
spells = flag.groupby(runs).agg(['first', 'size'])
hi = jpy_oos_k - 1
print('=== Flicker diagnostics (P(high regime) > 0.5) ===')
print(f'share of days with |change in probability| > 0.5   : {(ph.diff().abs() > 0.5).mean():.3f}')
print(f'mean length of high-regime spells (days)            : {spells.loc[spells["first"], "size"].mean():.1f}')
print(f'median model-implied expected duration, high (days) : {jpy_oos_log[f"exp_days_{hi}"].median():.1f}')
print(f'regime flips per year                               : {(flag != flag.shift()).sum() / (len(flag) / 252):.1f}')
print('Spells of only a few days mean the regime is catching single large returns, not sustained turbulence.')

### Forecast evaluation
Everything is scored on the same dates. QLIKE: lower is better. Diebold-Mariano and Mincer-Zarnowitz use HAC errors
because the 21-day targets overlap. Mincer-Zarnowitz: realised = a + b x forecast, well calibrated means a close to 0 and b close to 1.

In [ ]:
# ===== Cell F: evaluation (1-day and H-day average forecasts) =====
H = jpy_oos_H

def qlike(y, h):
    return np.log(h) + y / h

def dm_test(la, lb, lags):
    """Mean of (loss_a - loss_b); positive means model b has the lower loss. HAC (Newey-West) errors."""
    r = sm.OLS(np.asarray(la - lb), np.ones(len(la))).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    return r.params[0], r.tvalues[0], r.pvalues[0]

# ---------- H-day-average forecasts ----------
dH = pd.DataFrame({'Markov': jpy_oos_var_H, 'GARCH': expanding_forecast_variances_H,
                   'Naive 30d': jpy_oos_naive_var, 'y': jpy_oos_yH}).dropna()
dH['50/50 combo'] = 0.5 * (dH['Markov'] + dH['GARCH'])
names_H = ['GARCH', 'Markov', '50/50 combo', 'Naive 30d']
LH = pd.DataFrame({n: qlike(dH['y'], dH[n]) for n in names_H})
print(f'=== {H}-day-average forecasts: mean QLIKE, n = {len(dH):,} ({dH.index.min().date()} to {dH.index.max().date()}) ===')
print(LH.mean().round(4).to_string())
for a, b in [('GARCH', 'Markov'), ('GARCH', '50/50 combo'), ('Markov', '50/50 combo')]:
    d, t, pv = dm_test(LH[a], LH[b], H)
    print(f'DM {a} - {b}: mean diff {d:+.4f}  t={t:+.2f}  p={pv:.3f}   (positive = {b} better)')

print('\nMincer-Zarnowitz (HAC errors)')
for n in names_H:
    X = sm.add_constant(dH[[n]].rename(columns={n: 'h'}))
    r = sm.OLS(dH['y'], X).fit(cov_type='HAC', cov_kwds={'maxlags': H})
    pv = float(np.squeeze(r.wald_test('const = 0, h = 1', scalar=True).pvalue))
    print(f"{n:12s} a={r.params['const']:+.4f}  b={r.params['h']:.3f}  R2={r.rsquared:.3f}  p(a=0, b=1)={pv:.3f}")

print('\nBias by realised-variance tercile: mean forecast / mean realised (1.0 = unbiased)')
terc = pd.qcut(dH['y'], 3, labels=['calm', 'middle', 'turbulent'])
print(pd.DataFrame({n: dH.groupby(terc)[n].mean() / dH.groupby(terc)['y'].mean() for n in names_H}).round(2).to_string())

vr = jpy_oos_vix_regime.reindex(dH.index)
print('\nMean QLIKE by VIX regime (your High / Medium / Low thresholds):')
print(LH.groupby(vr).mean().reindex(['Low', 'Medium', 'High']).round(3).to_string())
print('\nMean QLIKE by calendar year:')
print(LH.groupby(LH.index.year).mean().round(3).to_string())

# ---------- 1-day-ahead forecasts ----------
d1 = pd.DataFrame({'Markov': jpy_oos_var, 'GARCH': expanding_forecast_variances,
                   'Naive 30d': jpy_oos_naive_var, 'y': jpy_oos_y1}).dropna()
L1 = pd.DataFrame({n: qlike(d1['y'], d1[n]) for n in ['GARCH', 'Markov', 'Naive 30d']})
print(f'\n=== 1-day-ahead forecasts vs squared return: mean QLIKE, n = {len(d1):,} ===')
print(L1.mean().round(4).to_string())
d, t, pv = dm_test(L1['GARCH'], L1['Markov'], 5)
print(f'DM GARCH - Markov: mean diff {d:+.4f}  t={t:+.2f}  p={pv:.3f}   (positive = Markov better)')

In [ ]:
# ===== Cell G: plots - H-day forecasts vs realised volatility, and where each model wins =====
ann = lambda v: np.sqrt(v * 252)
fig, ax = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
jpy_oos_shade(ax[0])
ax[0].plot(dH.index, ann(dH['y']), color='tab:orange', lw=0.9, label=f'realised vol over the forecast date + next {H - 1} days')
ax[0].plot(dH.index, ann(dH['GARCH']), color='tab:blue', lw=0.9, label=f'GARCH {H}-day forecast')
ax[0].plot(dH.index, ann(dH['Markov']), color='tab:purple', lw=0.9, label=f'Markov {H}-day forecast')
ax[0].set_ylabel('Annualised volatility (%)'); ax[0].legend(loc='upper right'); ax[0].grid(True, linestyle=':', alpha=0.5)
ax[0].set_title(f'{H}-day-average forecasts (made with data through D-1) vs realised volatility over D ... D+{H - 1}')
jpy_oos_shade(ax[1])
(LH['GARCH'] - LH['Markov']).cumsum().plot(ax=ax[1], color='black')
ax[1].axhline(0, color='grey', lw=0.8); ax[1].grid(True, linestyle=':', alpha=0.5)
ax[1].set_title('Cumulative QLIKE difference (GARCH - Markov): rising = Markov better, falling = GARCH better')
handles = [Patch(facecolor=jpy_oos_colors[n], alpha=0.2, label=f'{n} VIX regime') for n in ['Low', 'Medium', 'High']]
ax[1].legend(handles=handles, loc='upper left')
plt.tight_layout(); plt.show()

### Simulation: what α+β does a GARCH report when the truth is two fixed variance levels?
The simulated returns contain **no** GARCH dynamics: inside each regime they are plain i.i.d. normal noise. If a GARCH(1,1)
still reports α+β close to 1, that is the Lamoureux-Lastrapes effect. The Markov model is fitted with your spec
(switching mean and variance) to check that it recovers the true volatilities.

In [ ]:
# ===== Cell H: simulation =====
def simulate_two_regime(n=5000, sd=(0.4, 1.0), stay=(0.99, 0.97), seed=0):
    rng = np.random.default_rng(seed)
    state = np.zeros(n, int)
    u = rng.random(n)
    for t in range(1, n):
        state[t] = state[t - 1] if u[t] < stay[state[t - 1]] else 1 - state[t - 1]
    return pd.Series(rng.standard_normal(n) * np.asarray(sd)[state])

if jpy_oos_run_sim:
    out = []
    for seed in range(10):
        r_sim = simulate_two_regime(seed=seed)
        g = arch_model(r_sim, mean='Constant', vol='Garch', p=1, q=1, dist='normal').fit(disp='off')
        m = MarkovRegression(r_sim, k_regimes=2, switching_variance=True).fit(search_reps=10, disp=False)
        sg = np.sort(np.sqrt([m.params['sigma2[0]'], m.params['sigma2[1]']]))
        out.append({'seed': seed, 'garch_alpha+beta': g.params['alpha[1]'] + g.params['beta[1]'],
                    'ms_sigma_low': sg[0], 'ms_sigma_high': sg[1]})
    sim = pd.DataFrame(out)
    print('True daily sd = 0.4 (calm) / 1.0 (high); true stay probabilities 0.99 / 0.97')
    print(sim.round(3).to_string(index=False))
    print(sim[['garch_alpha+beta', 'ms_sigma_low', 'ms_sigma_high']].mean().round(3).to_string())